# 03 — Analysis & Figures

Produces the three core figures for the paper:
- **Figure 1** — Monthly coverage volume (RQ1)
- **Figure 2** — Frame distribution over time (RQ2)
- **Figure 3** — Event study around the EU AI Act (RQ3)
- **Figure 4** (optional) — Regional frame comparison

Figures are saved to `../paper/figures/`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd

sys.path.insert(0, str(Path('..')))

from src.analysis import monthly_volume, frame_shares, event_study, region_comparison
from src.dictionaries import MILESTONES

FIGURES = Path('../paper/figures')
FIGURES.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='colorblind')

In [ ]:
df = pd.read_parquet('../data/interim/gdelt_preprocessed.parquet')
print(f'Loaded {len(df):,} rows')

## Figure 1 — Monthly coverage volume (RQ1)

In [ ]:
vol = monthly_volume(df)
vol_dates = vol.index.to_timestamp()

milestone_dates = {m['name']: pd.Timestamp(m['date']) for m in MILESTONES}

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(vol_dates, vol.values, linewidth=1.5)

for name, ts in milestone_dates.items():
    ax.axvline(ts, color='grey', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.text(ts, ax.get_ylim()[1] * 0.92, name.replace('_', ' '), rotation=90,
            fontsize=7, ha='right', va='top', color='grey')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')
ax.set_xlabel('Month')
ax.set_ylabel('Number of articles')
ax.set_title('Monthly volume of generative AI governance news in GDELT GKG')
plt.tight_layout()
plt.savefig(FIGURES / 'fig1_monthly_volume.pdf', bbox_inches='tight')
plt.show()

## Figure 2 — Frame distribution over time (RQ2)

In [ ]:
shares = frame_shares(df)
shares_dates = shares.index.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(
    shares_dates,
    [shares[col] for col in shares.columns],
    labels=[c.replace('_', ' ').title() for c in shares.columns],
    alpha=0.85,
)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45, ha='right')
ax.set_xlabel('Month')
ax.set_ylabel('Frame share')
ax.set_ylim(0, 1)
ax.set_title('Governance frame distribution over time')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(FIGURES / 'fig2_frame_shares.pdf', bbox_inches='tight')
plt.show()

## Figure 3 — Event study: EU AI Act (RQ3)

In [ ]:
es = event_study(df, 'eu_ai_act', window=3)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: volume
axes[0].bar(es['volume'].index, es['volume'].values)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Months relative to EU AI Act adoption')
axes[0].set_ylabel('Article count')
axes[0].set_title('Coverage volume around EU AI Act')

# Right: frame shares
es['shares'].plot(kind='bar', stacked=True, ax=axes[1], legend=True)
axes[1].axvline(3, color='red', linestyle='--', linewidth=1)  # event month is at index 3
axes[1].set_xticklabels([str(i) for i in range(-3, 4)], rotation=0)
axes[1].set_xlabel('Months relative to EU AI Act adoption')
axes[1].set_ylabel('Frame share')
axes[1].set_title('Frame distribution around EU AI Act')
axes[1].legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig(FIGURES / 'fig3_event_study_eu_ai_act.pdf', bbox_inches='tight')
plt.show()

## Figure 4 (optional) — Regional comparison

In [ ]:
reg = region_comparison(df)

fig, ax = plt.subplots(figsize=(8, 4))
reg.plot(kind='bar', stacked=True, ax=ax)
ax.set_xlabel('Region')
ax.set_ylabel('Frame share')
ax.set_title('Frame distribution by region (US / EU / UK)')
ax.legend(fontsize=8, loc='upper right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES / 'fig4_regional_comparison.pdf', bbox_inches='tight')
plt.show()